# D5 · xBD converter check and the extra xBD tiles (CPU, no GPU)

Task **D5** from `Essential_task_delegation/Next_Steps_Roadmap.md`. It prepares the extra training data for **R12** and measures how DisasterM3 mapped xBD's five damage labels.

**Status (18 Oct 2026): done. This notebook was run on a PC (local mode), and the cell outputs below are from that run.**
- **Inputs on the PC:**
  - DisasterM3's masks from Hugging Face (`Kingdrone-Junjue/DisasterM3`, `DisasterM3_Instruct/masks.zip`, the same files as the Kaggle mirror), unzipped into `dataset/DisasterM3 Dataset/DisasterM3_Instruct/masks/`, which is git-ignored;
  - the xBD labels in `data/`.
- **Outputs:** `Champion_Task/D5-results/`.
- **Counts:** they match Phase2_Plan: 1,792 overlapping tiles, 1,007 new ones, 3,472 destroyed polygons.
- **Mapping:**
  - no-damage → Intact, minor and major → Damaged, destroyed → Destroyed. Each keeps about 98% of its pixels; the rest is polygon edges.
  - DisasterM3 left **un-classified** buildings as background (99.96%), so they're ignored in training.
  - Lookup table: `[0, 1, 2, 2, 3, 255]`.
- **Agreement after mapping:** 97.8% of building pixels; median building IoU per tile 0.929.
- **Next step:** upload `D5-results/` as the Kaggle dataset `xbd-extra-1007` for R12.

**What it does**
1. **Tile IDs:** finds which of xBD's 2,799 post-disaster training tiles DisasterM3 already uses (1,792) and which are new (1,007). It also flags the new tiles that are DisasterM3 **benchmark** images.
2. **Converter check:** draws the xBD polygons of the overlapping tiles and compares them pixel by pixel with DisasterM3's own 4-class masks (built as in the D4 notebooks). The result is a table of xBD label against DisasterM3 class, and the label mapping R12 should use.
3. **Converts the new tiles** into masks.

**Leakage warning (found while building this notebook)**
- 447 of the 1,007 new tiles are DisasterM3 benchmark images, and they hold almost all the destroyed buildings (3,466 of 3,472 destroyed polygons).
- None of them is in our 645 validation images, so training on them is fine for **O1** (segmentation scores).
- But 741 of the 4,982 BDC benchmark questions use these images. A U-Net trained on them must not be used for **H1/KR2** counting, unless those questions are scored separately.
- Every tile in `manifest.json` carries `in_benchmark`, so R12 can use either pool. The benchmark-safe pool has 560 tiles but only 6 destroyed polygons.

**How to run**
- **On a PC (as done here):** open this notebook in VS Code or Jupyter and Run All. Cell 1 switches to local paths when `/kaggle/input` doesn't exist. It needs numpy, pandas, Pillow and tqdm only, and about 3 minutes.
- **On Kaggle:**
  1. Use a CPU session with internet off.
  2. Attach the DisasterM3 mirror (attached automatically), the xBD labels uploaded as a private dataset (`data/xView2 Challenge Dataset - train and test/train/labels/`, 319 MB), and `benchmark_release.json`: either attach `redwannewaz/disasterm3-bench-mirror-v2` or add the local copy to the labels upload. Cell 1 finds both by file name.
  3. Run with `SMOKE_TEST = True` first, then Save & Run All. Outputs go to `/kaggle/working/D5/`.

**Outputs:**
- `masks/` (one PNG per new tile, raw xBD codes 0–5);
- `manifest.json`, `mapping.json`, `overlap_check.csv`;
- `mapping_report.md`, `config.json`, `README.md`.

**Done when:**
- the mapping table is clear (expected: minor and major become Damaged);
- no new tile is in DisasterM3's segmentation data (asserted in Cell 2);
- the counts match Phase2_Plan §2 (1,007 tiles, 3,472 destroyed polygons).

In [1]:
# ── Cell 1: Configuration and inputs (CPU only, no installs needed) ─────────
import glob
import json
import re
import time
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

SMOKE_TEST = False                  # True = 50 overlapping and 20 new tiles, about a minute
ON_KAGGLE = Path("/kaggle/input").exists()

def find_input(pattern):
    """First file under /kaggle/input that matches, whatever the dataset is called."""
    hits = sorted(glob.glob(f"/kaggle/input/**/{pattern}", recursive=True))
    if not hits:
        raise FileNotFoundError(f"No '{pattern}' under /kaggle/input. Attach it as described in the first cell.")
    return Path(hits[0])

if ON_KAGGLE:
    DATA_ROOT = Path("/kaggle/input/datasets/abrarmohammedtanzim/disasterm3-mirror/DisasterM3_Instruct")
    MANIFEST_PATH = DATA_ROOT / "train_release.json"
    OUT_DIR = Path("/kaggle/working/D5")
    LABELS_DIR = find_input("*_post_disaster.json").parent    # the uploaded xBD train/labels folder
    BENCH_PATH = find_input("benchmark_release.json")
else:
    # This PC: the repo root is the folder that holds train_release.json. DisasterM3's masks come from
    # Hugging Face (Kingdrone-Junjue/DisasterM3, DisasterM3_Instruct/masks.zip), unzipped into a git-ignored folder.
    REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "train_release.json").exists())
    DATA_ROOT = REPO / "dataset" / "DisasterM3 Dataset" / "DisasterM3_Instruct"
    MANIFEST_PATH = REPO / "train_release.json"
    OUT_DIR = REPO / "Essential_task_delegation" / "Champion_Task" / "D5-results"
    LABELS_DIR = REPO / "data" / "xView2 Challenge Dataset - train and test" / "train" / "labels"
    BENCH_PATH = REPO / "support" / "Abrar" / "Implementation_DisasterM3" / "benchmark_release.json"

(OUT_DIR / "masks").mkdir(parents=True, exist_ok=True)
t_start = time.time()
print(f"✓ D5 xBD converter on {'Kaggle' if ON_KAGGLE else 'this PC'}{'  [SMOKE TEST]' if SMOKE_TEST else ''}")
print(f"  DisasterM3:  {DATA_ROOT}")
print(f"  xBD labels:  {LABELS_DIR}")
print(f"  Benchmark:   {BENCH_PATH}")
print(f"  Outputs:     {OUT_DIR}")

✓ D5 xBD converter on this PC
  DisasterM3:  D:\CSE499AB_project\dataset\DisasterM3 Dataset\DisasterM3_Instruct
  xBD labels:  D:\CSE499AB_project\data\xView2 Challenge Dataset - train and test\train\labels
  Benchmark:   D:\CSE499AB_project\support\Abrar\Implementation_DisasterM3\benchmark_release.json
  Outputs:     D:\CSE499AB_project\Essential_task_delegation\Champion_Task\D5-results


## 1. Tile IDs: which xBD tiles DisasterM3 already uses
DisasterM3 writes the event with underscores (`hurricane_florence_00000261_post_disaster.png`); xBD uses dashes (`hurricane-florence_00000261`). Both become the same tile ID. A tile is **in DisasterM3** if its pre or post image appears in the segmentation entries of `train_release.json`, which also hold the 645 validation images. The benchmark is checked separately, because H1 scores KR2 on it.

In [2]:
# ── Cell 2: Tile IDs: which xBD tiles DisasterM3 already uses ───────────────
XBD_NAME = re.compile(r"^(.+?)_(\d{8})_(?:pre|post)_disaster")

def tile_id(path):
    """'train_images\\hurricane_florence_00000261_post_disaster.png' -> 'hurricane-florence_00000261'.
    None for images that aren't xBD tiles."""
    m = XBD_NAME.match(Path(str(path or "").replace("\\", "/")).name.replace("_sar", ""))
    return f"{m.group(1).replace('_', '-')}_{m.group(2)}" if m else None

def tiles_in(entries):
    return {tile_id(e.get(k)) for e in entries for k in ("pre_image_path", "post_image_path")} - {None}

with open(MANIFEST_PATH, encoding="utf-8") as f:
    train_entries = json.load(f)
with open(BENCH_PATH, encoding="utf-8") as f:
    bench_entries = json.load(f)
bda = [e for e in train_entries if e.get("task") == "Referring Expression Segmentation"
       and e.get("cls_description") == "Building Damage Assessment"]
seg_tiles = tiles_in(bda)              # the U-Net's 6,443 pairs, including the 645 validation images
bench_tiles = tiles_in(bench_entries)  # the benchmark that H1 scores for KR2

xbd_tiles = sorted(p.name[:-len("_post_disaster.json")] for p in LABELS_DIR.glob("*_post_disaster.json"))
overlap = [t for t in xbd_tiles if t in seg_tiles]
new = [t for t in xbd_tiles if t not in seg_tiles]
assert not set(new) & seg_tiles        # so no new tile can be one of the 645 validation images

new_in_bench = {t for t in new if t in bench_tiles}
bdc = [e for e in bench_entries if e.get("task") == "Building Damage Counting"]
bdc_hit = sum(tile_id(e.get("post_image_path")) in new_in_bench for e in bdc)
print(f"xBD post-disaster labels: {len(xbd_tiles):,}")
print(f"  already in DisasterM3's segmentation data: {len(overlap):,} (expected 1,792)")
print(f"  new:                                       {len(new):,} (expected 1,007)")
print(f"    of which DisasterM3 benchmark images:    {len(new_in_bench):,} "
      f"({bdc_hit:,} of {len(bdc):,} BDC benchmark questions use them)")
if SMOKE_TEST:
    overlap, new = overlap[:50], new[:20]

xBD post-disaster labels: 2,799
  already in DisasterM3's segmentation data: 1,792 (expected 1,792)
  new:                                       1,007 (expected 1,007)
    of which DisasterM3 benchmark images:    447 (741 of 4,982 BDC benchmark questions use them)


## 2. Converter check
xBD stores each building as a polygon in pixel coordinates, with one of five labels. They're drawn into a mask with codes 0 (background), 1 no-damage, 2 minor-damage, 3 major-damage, 4 destroyed and 5 un-classified; worse damage is drawn last. On the overlapping tiles this mask is compared with DisasterM3's 4-class mask.

Each xBD label maps to the DisasterM3 building class it mostly becomes. `un-classified` becomes 255 (ignored in training) unless DisasterM3 consistently gives it one class.

In [3]:
# ── Cell 3: xBD polygons -> mask with one code per damage label ────────────
from PIL import Image, ImageDraw     # PIL, not cv2, so a local run and a Kaggle run give identical masks

SUBTYPES = ["no-damage", "minor-damage", "major-damage", "destroyed", "un-classified"]
CODE = {s: i + 1 for i, s in enumerate(SUBTYPES)}       # 0 = background, then 1-5 in this order
XBD_CODES = ["background", *SUBTYPES]
DRAW_ORDER = ["un-classified", "no-damage", "minor-damage", "major-damage", "destroyed"]  # worst damage last

def rasterize(label):
    """xBD label JSON -> (uint8 mask with codes 0-5, polygon count per label).
    Every polygon in xBD's train labels is a single ring: 'POLYGON ((x y, x y, ...))'."""
    h, w = label["metadata"].get("height", 1024), label["metadata"].get("width", 1024)
    polys = defaultdict(list)
    for f in label["features"]["xy"]:
        sub = f["properties"].get("subtype")
        if sub in CODE:
            wkt = f["wkt"]
            ring = wkt[wkt.index("((") + 2:wkt.rindex("))")]
            pts = np.round([[float(v) for v in p.split()] for p in ring.split(",")]).astype(int)
            polys[sub].append([tuple(pt) for pt in pts])
    img = Image.new("L", (w, h), 0)
    draw = ImageDraw.Draw(img)
    for sub in DRAW_ORDER:
        for p in polys[sub]:           # one polygon at a time, so a later one overwrites an earlier one
            draw.polygon(p, fill=CODE[sub])
    return np.array(img), {s: len(polys[s]) for s in SUBTYPES}

def load_label(tile):
    with open(LABELS_DIR / f"{tile}_post_disaster.json", encoding="utf-8") as f:
        return json.load(f)

# Self-check: a destroyed square inside a no-damage square keeps both labels
_toy = {"metadata": {"width": 20, "height": 20}, "features": {"xy": [
    {"properties": {"subtype": "destroyed"}, "wkt": "POLYGON ((5 5, 9 5, 9 9, 5 9, 5 5))"},
    {"properties": {"subtype": "no-damage"}, "wkt": "POLYGON ((2 2, 12 2, 12 12, 2 12, 2 2))"}]}}
_m, _n = rasterize(_toy)
assert _m[7, 7] == CODE["destroyed"] and _m[3, 3] == CODE["no-damage"] and _m[15, 15] == 0
assert _n["destroyed"] == 1 and _n["no-damage"] == 1
print("✓ Rasterizer ready (self-check passed)")

✓ Rasterizer ready (self-check passed)


In [4]:
# ── Cell 4: Converter check against DisasterM3's own masks ─────────────────
# DisasterM3's masks are found and combined exactly as in Cell 4 of the D4 notebooks (read with PIL here).
def resolve_path(rel_path):
    if not rel_path:
        return None
    rel_path = rel_path.replace("\\", "/")
    filename = Path(rel_path).name
    candidates = [
        DATA_ROOT / rel_path,
        DATA_ROOT / "train_images" / rel_path,
        DATA_ROOT / "train_images" / "train_images" / filename,
        DATA_ROOT / "masks" / rel_path,
        DATA_ROOT / "masks" / "masks" / filename,
    ]
    for c in candidates:
        if c.exists():
            return str(c)
    return None

FOLDER_TO_CLASS = {
    "train_building_intact_mask": 1,
    "train_building_damaged_mask": 2,
    "train_building_destroyed_mask": 3,
}
overlap_set = set(overlap)
dm3_mask_paths = defaultdict(dict)     # tile -> {mask folder: path in the manifest}
for e in bda:
    t = tile_id(e.get("post_image_path"))
    if t in overlap_set and e.get("image_type") == "Optical":
        folder = Path(e.get("ground_truth", "").replace("\\", "/")).parent.name
        dm3_mask_paths[t][folder] = e.get("ground_truth")

def dm3_mask(tile):
    """DisasterM3's 4-class mask (0 background, 1 intact, 2 damaged, 3 destroyed); later classes overwrite."""
    combined = None
    for folder, cls in FOLDER_TO_CLASS.items():
        path = resolve_path(dm3_mask_paths[tile].get(folder))
        if path is None:
            continue
        m = np.array(Image.open(path).convert("L"))
        if combined is None:
            combined = np.zeros_like(m, dtype=np.uint8)
        combined[m > 0] = cls
    return combined

DM3_CLASSES = ["Background", "Intact", "Damaged", "Destroyed"]
table = np.zeros((len(XBD_CODES), len(DM3_CLASSES)), np.int64)   # rows: xBD code, columns: DisasterM3 class
check_rows, no_dm3_mask = [], 0
for t in tqdm(overlap, desc="Overlapping tiles"):
    dm3 = dm3_mask(t)
    if dm3 is None:
        no_dm3_mask += 1
        continue
    xbd, _ = rasterize(load_label(t))
    if xbd.shape != dm3.shape:
        xbd = np.array(Image.fromarray(xbd).resize((dm3.shape[1], dm3.shape[0]), Image.NEAREST))
    cm = np.bincount(xbd.ravel().astype(np.int64) * 4 + dm3.ravel(), minlength=24).reshape(6, 4)
    table += cm
    union = cm[1:].sum() + cm[0, 1:].sum()        # building pixels in either mask
    check_rows.append({"tile": t, "event": t.rsplit("_", 1)[0],
                       "building_iou": round(cm[1:, 1:].sum() / union, 4) if union else 1.0})
check = pd.DataFrame(check_rows)
assert len(check), "No DisasterM3 masks found for the overlapping tiles: is the mirror attached?"

# Each xBD label maps to the DisasterM3 building class it mostly becomes
lut, mapping_rows = [0], []
for c in range(1, len(XBD_CODES)):
    row = table[c]
    building_share = row[1:].sum() / max(row.sum(), 1)   # how often DisasterM3 draws it as a building at all
    cls = int(row[1:].argmax()) + 1
    class_share = row[cls] / max(row[1:].sum(), 1)       # how consistently it gets that one class
    ignore = XBD_CODES[c] == "un-classified" and (building_share < 0.5 or class_share < 0.8)
    lut.append(255 if ignore else cls)
    mapping_rows.append({"xbd_label": XBD_CODES[c], "pixels": int(row.sum()),
                         **{f"to_{n.lower()}": round(row[i] / max(row.sum(), 1), 4)
                            for i, n in enumerate(DM3_CLASSES)},
                         "maps_to": "255 (ignore)" if ignore else DM3_CLASSES[cls],
                         "consistency": round(class_share, 4)})
mapping = pd.DataFrame(mapping_rows)
agreement = sum(table[c, lut[c]] for c in range(1, 5)) / max(table[1:5].sum(), 1)   # the four graded labels

print(f"Compared {len(check):,} tiles ({no_dm3_mask} had no DisasterM3 mask)")
print(mapping.to_string(index=False))
print(f"\nPixel agreement after mapping (graded buildings): {agreement:.1%}")
print(f"Median building IoU per tile, xBD drawing vs DisasterM3: {check['building_iou'].median():.3f} "
      f"({(check['building_iou'] < 0.5).sum()} tiles below 0.5)")
for r in mapping_rows:
    if r["xbd_label"] != "un-classified" and r["consistency"] < 0.8:
        print(f"⚠ DisasterM3 is not consistent for {r['xbd_label']}: check the table before training")
print(f"Lookup table for R12 (xBD code 0-5 -> class): {lut}")

Overlapping tiles:   0%|          | 0/1792 [00:00<?, ?it/s]

Compared 1,792 tiles (0 had no DisasterM3 mask)
    xbd_label   pixels  to_background  to_intact  to_damaged  to_destroyed      maps_to  consistency
    no-damage 72900780         0.0212     0.9788      0.0000         0.000       Intact       1.0000
 minor-damage 14189745         0.0224     0.0001      0.9775         0.000      Damaged       0.9999
 major-damage 17281674         0.0205     0.0001      0.9794         0.000      Damaged       0.9999
    destroyed  6848976         0.0267     0.0001      0.0002         0.973    Destroyed       0.9997
un-classified  1096413         0.9996     0.0002      0.0001         0.000 255 (ignore)       0.6370

Pixel agreement after mapping (graded buildings): 97.8%
Median building IoU per tile, xBD drawing vs DisasterM3: 0.929 (14 tiles below 0.5)
Lookup table for R12 (xBD code 0-5 -> class): [0, 1, 2, 2, 3, 255]


## 3. Convert the new tiles and write the outputs
Masks keep the raw xBD codes 0–5, so the mapping can change later without re-running this notebook. R12 applies the lookup table from `mapping.json` when it loads a mask, and chooses its pool with `in_benchmark` from `manifest.json`:
- **all new tiles:** fine for segmentation scores (O1);
- **benchmark-safe** (`in_benchmark = false`): also safe for H1/KR2 counting.

In [5]:
# ── Cell 5: Convert the new tiles (raw xBD codes 0-5) ──────────────────────
manifest = []
for t in tqdm(new, desc="New tiles"):
    mask, polygons = rasterize(load_label(t))
    Image.fromarray(mask).save(OUT_DIR / "masks" / f"{t}_post_disaster.png")
    pixels = np.bincount(mask.ravel(), minlength=len(XBD_CODES))
    manifest.append({
        "tile": t, "event": t.rsplit("_", 1)[0], "in_benchmark": t in bench_tiles,
        "post_image": f"{t}_post_disaster.png", "pre_image": f"{t}_pre_disaster.png",
        "mask": f"masks/{t}_post_disaster.png",
        "polygons": polygons, "pixels": {XBD_CODES[c]: int(pixels[c]) for c in range(len(XBD_CODES))},
    })
print(f"✓ Wrote {len(manifest):,} masks to {OUT_DIR / 'masks'}")

New tiles:   0%|          | 0/1007 [00:00<?, ?it/s]

✓ Wrote 1,007 masks to D:\CSE499AB_project\Essential_task_delegation\Champion_Task\D5-results\masks


In [6]:
# ── Cell 6: The two R12 pools, and the output files ────────────────────────
def as_markdown(df):
    try:
        return df.to_markdown(index=False)
    except ImportError:              # `tabulate` missing: fall back to a plain-text table
        return "```\n" + df.to_string(index=False) + "\n```"

tiles = pd.DataFrame([{"tile": m["tile"], "event": m["event"], "in_benchmark": m["in_benchmark"],
                       "destroyed_polygons": m["polygons"]["destroyed"]} for m in manifest])
pools = pd.DataFrame([
    {"pool": name, "tiles": len(df), "destroyed_polygons": int(df["destroyed_polygons"].sum()),
     "tiles_with_destroyed": int((df["destroyed_polygons"] > 0).sum())}
    for name, df in [("all new tiles", tiles),
                     ("benchmark-safe (in_benchmark = false)", tiles[~tiles["in_benchmark"]])]])
per_event = (tiles.groupby("event")
             .agg(tiles=("tile", "size"), in_benchmark=("in_benchmark", "sum"),
                  destroyed_polygons=("destroyed_polygons", "sum"))
             .sort_values("tiles", ascending=False).reset_index())
print(pools.to_string(index=False))
print()
print(per_event.to_string(index=False))

with open(OUT_DIR / "manifest.json", "w") as f:
    json.dump(manifest, f, indent=1)
with open(OUT_DIR / "mapping.json", "w") as f:
    json.dump({"lut": lut, "codes": XBD_CODES, "classes": DM3_CLASSES, "table": table.tolist(),
               "pixel_agreement": round(float(agreement), 4)}, f, indent=1)
check.to_csv(OUT_DIR / "overlap_check.csv", index=False)

(OUT_DIR / "mapping_report.md").write_text("\n".join([
    "# D5 mapping report", "",
    f"{len(check):,} overlapping tiles: xBD polygons drawn by this notebook against DisasterM3's own masks. "
    "Each row shows where one xBD label's pixels go in DisasterM3.", "",
    as_markdown(mapping), "",
    f"Pixel agreement after mapping: {agreement:.1%}. "
    f"Median building IoU per tile: {check['building_iou'].median():.3f}.", "",
    f"Lookup table (xBD code 0-5 -> class, 255 = ignore): `{lut}`",
]), encoding="utf-8")

config = {"task": "D5", "smoke_test": SMOKE_TEST, "labels_dir": str(LABELS_DIR), "benchmark": str(BENCH_PATH),
          "xbd_post_labels": len(xbd_tiles), "overlap_tiles_checked": len(check), "new_tiles": len(manifest),
          "new_in_benchmark": len(new_in_bench), "bdc_questions_on_them": bdc_hit, "lut": lut,
          "run_time_min": round((time.time() - t_start) / 60, 1)}
(OUT_DIR / "config.json").write_text(json.dumps(config, indent=1), encoding="utf-8")

(OUT_DIR / "README.md").write_text("\n".join([
    "# xbd-extra-1007 (task D5)", "",
    f"{len(manifest):,} xBD training tiles that are not in DisasterM3's segmentation data, so none of them "
    "is one of the 645 validation images.", "",
    "## Pools for R12", "",
    as_markdown(pools), "",
    f"{len(new_in_bench):,} tiles are DisasterM3 benchmark images ({bdc_hit:,} of {len(bdc):,} BDC questions "
    "use them). Training on them is fine for segmentation scores (O1). A model trained on them must not be "
    "used for H1/KR2 counting unless those questions are scored separately.", "",
    "## Use in R12", "",
    "Masks keep the raw xBD codes (0 background, 1 no-damage, 2 minor, 3 major, 4 destroyed, 5 un-classified). "
    "Map them when loading:", "",
    "```python",
    "D5 = Path('/kaggle/input/xbd-extra-1007')   # wherever the dataset is mounted",
    "lut = np.array(json.load(open(D5 / 'mapping.json'))['lut'], np.uint8)",
    "mask = lut[cv2.imread(str(D5 / m['mask']), cv2.IMREAD_GRAYSCALE)]   # 0-3, 255 = ignore",
    "```", "",
    "Images (`post_image` and `pre_image` in `manifest.json`) come from xBD's `train/images` folder.", "",
    "## Mapping", "", f"See `mapping_report.md`. Lookup table: `{lut}`.", "",
    "## Per event", "", as_markdown(per_event),
]), encoding="utf-8")
print(f"\n✓ Wrote manifest.json, mapping.json, overlap_check.csv, mapping_report.md, config.json and README.md "
      f"({(time.time() - t_start) / 60:.1f} min)")
print("Next: create a Kaggle dataset from this output called xbd-extra-1007 (used by R12).")

                                 pool  tiles  destroyed_polygons  tiles_with_destroyed
                        all new tiles   1007                3472                   163
benchmark-safe (in_benchmark = false)    560                   6                     3

              event  tiles  in_benchmark  destroyed_polygons
         socal-fire    467            27                  81
  mexico-earthquake    121           121                   2
       palu-tsunami     82            74                2206
   midwest-flooding     65            29                   5
   hurricane-harvey     61            39                  31
  hurricane-matthew     56            38                 227
 hurricane-florence     51            33                   0
  hurricane-michael     45            35                  99
santa-rosa-wildfire     41            34                 796
  guatemala-volcano     18            17                  25

✓ Wrote manifest.json, mapping.json, overlap_check.csv, mapping_re